# 02 Collect Sources

Collect and audit the dynamic source artifacts used by the inland Event Catalog: USGS active streamgages, direct AORC SST rainfall members, and NWM soil-moisture context.


> **Stage Contract**
>
> Requires: region setup outputs, provider credentials where required
>
> Produces: USGS streamgage candidates and reviewed discharge records, AORC SST rainfall, and NWM soil-moisture source artifacts
>
> Next: 03_build_event_catalog.ipynb


In [ ]:
import sys
from pathlib import Path
import importlib

import pandas as pd
from IPython.display import display

location_root = Path.cwd().parent
repo_root = location_root.parents[1]
src_root = repo_root / "src"
src_path = str(src_root)
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
importlib.invalidate_caches()

import design_events.collect_sources.notebook as collect
collect = importlib.reload(collect)

source_skip_existing = True
write_reviewed_streamgage_network_file = True

runtime = collect.load_collect_sources_notebook_runtime(location_root)
config = runtime.runtime_config
paths = runtime.runtime_paths

display(collect.source_collection_runtime_summary(config, paths))
display(collect.source_record_location_table(runtime))


## Source Collection Plan

Review the configured source contracts before starting provider calls. This is the first place a stakeholder should see whether a source is required, optional, disabled, or ready for the full inland production window.


In [ ]:
plan = collect.build_source_collection_plan(config, paths)
plan_table = collect.source_collection_plan_with_reuse_table(
    plan,
    paths,
    source_skip_existing=source_skip_existing,
)
display(plan_table)


## USGS Active Streamgages

USGS active streamgage discovery replaces the Marshfield coastal water-level and wave source blocks. The candidate GeoJSON is only the discovery layer; the reviewed network below is the auditable set used for event frequency, validation, and Wflow/SFINCS handoff context.


In [ ]:
discovery_summary, records_summary = collect.usgs_streamgage_discovery_summary(runtime)
display(discovery_summary)
display(records_summary)
display(collect.streamgage_source_readiness(runtime))


## Streamgage Review Gate

The HUC-region review converts active USGS candidates inside the selected Wflow watershed into an explicit model-facing network before discharge records are collected for the event catalog.


In [ ]:
display(collect.streamgage_review_summary(runtime))
display(collect.streamgage_review_source_table(runtime))


## Stochastic Storm Transposition Region

The SST region is the configured AORC SST transposition polygon, not a notebook-specific hand draw. For inland sites, it should cover the selected watershed/evaluation footprint so rainfall members are hydrologically plausible without drifting into an unrelated regional climate.


In [ ]:
# Plot the configured AORC SST transposition region before pulling rainfall members.
def _location_path(value):
    path = Path(value)
    if path.is_absolute():
        return path
    if path.parts and path.parts[0] in {"data", "02_flood", "01_grid"}:
        return paths["location_root"] / path
    return paths["repo_root"] / path


def plot_sst_region(config, paths, zoom=9, basemap=True):
    import geopandas as gpd
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    try:
        import contextily as ctx
    except ImportError:
        ctx = None

    region_file = config["collection"]["aorc_sst"]["transposition_region"]["geometry_file"]
    footprint_file = config["grid_footprint"]["source"]
    region_path = _location_path(region_file)
    footprint_path = _location_path(footprint_file)

    display(pd.Series({
        "transposition_region": region_file,
        "transposition_region_exists": region_path.exists(),
        "grid_footprint": footprint_file,
        "grid_footprint_exists": footprint_path.exists(),
        "plot_crs": "EPSG:3857",
        "analysis_crs": "EPSG:4326",
    }, name="aorc_sst_region_inputs"))

    region = gpd.read_file(region_path).to_crs("EPSG:4326")
    footprint = gpd.read_file(footprint_path).to_crs("EPSG:4326")
    region_web = region.to_crs(epsg=3857)
    footprint_web = footprint.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(8, 8))
    region_web.plot(ax=ax, facecolor="#d95f0226", edgecolor="#d95f02", linewidth=2.5)
    footprint_web.boundary.plot(ax=ax, color="#1b9e77", linewidth=2.0)

    xmin, ymin, xmax, ymax = region_web.total_bounds
    pad_x = (xmax - xmin) * 0.05
    pad_y = (ymax - ymin) * 0.05
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)

    if basemap and ctx is not None:
        try:
            ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, zoom=zoom, attribution_size=7)
        except Exception as exc:
            ax.text(0.01, 0.01, f"basemap unavailable: {type(exc).__name__}", transform=ax.transAxes, fontsize=8)

    ax.legend(
        handles=[
            Patch(facecolor="#d95f0226", edgecolor="#d95f02", label="Storm transposition region"),
            Line2D([0], [0], color="#1b9e77", linewidth=2.0, label="Study grid footprint"),
        ],
        loc="lower left",
    )
    ax.set_axis_off()
    ax.set_title(f"{paths['location_name'].title()} stochastic storm transposition region")
    fig.tight_layout()
    return fig, ax


fig, ax = plot_sst_region(config, paths)


## Direct AORC SST Rainfall Members

The direct AORC SST collector scans the transposition region, ranks rolling storm windows, and writes the rainfall-member table used to pair precipitation with streamflow and antecedent soil-moisture states.


In [ ]:
aorc_sst = config["collection"].get("aorc_sst", {})

display(pd.Series({
    "source": "direct_aorc_sst",
    "transposition_region_id": aorc_sst.get("transposition_region", {}).get("id"),
    "transposition_region": aorc_sst.get("transposition_region", {}).get("geometry_file"),
    "start_date": aorc_sst.get("start_date", config["collection"].get("start")),
    "end_date": aorc_sst.get("end_date", config["collection"].get("end")),
    "storm_duration_hours": aorc_sst.get("storm_duration_hours", 72),
    "top_n_events": aorc_sst.get("top_n_events"),
    "check_every_n_hours": aorc_sst.get("check_every_n_hours"),
    "decluster_hours": aorc_sst.get("decluster_hours"),
    "rainfall_members": str(paths["aorc_sst_rainfall_members_csv"]),
    "rainfall_members_exists": paths["aorc_sst_rainfall_members_csv"].exists(),
}, name="aorc_sst"))


## NWM Soil-Moisture Context

Selected NWM LDAS soil-moisture cells are retained for antecedent-condition pairing. Inland streamflow frequency uses reviewed USGS gages, so NWM streamflow remains context rather than the production frequency source.


In [ ]:
display(collect.nwm_soil_moisture_source_summary(config, paths))


## Run Collection

Run each configured source inside this notebook and show progress as each provider finishes. This cell is configured for the full inland production collection window, so it overwrites partial smoke artifacts instead of reusing them.


In [ ]:
# Collect configured sources and summarize the artifacts.
run_collection = True
collection_parameters = pd.Series({
    "run_collection": run_collection,
    "skip_existing": source_skip_existing,
    "stop_on_error": True,
    "sources": ", ".join(plan.source_names),
}, name="collection_run_parameters")
display(collection_parameters)

collection_result_table = collect.run_collect(
    config,
    paths,
    plan,
    run_collection=run_collection,
    skip_existing=source_skip_existing,
)
display(collection_result_table)


## Reviewed Streamgage Network Writer

Write the reviewed GeoJSON from candidate gages and the source-owned HUC review method. This keeps the notebook auditable: the decision table shows which gages are accepted before discharge records are fetched.


In [ ]:
reviewed_network_write = collect.build_or_write_reviewed_streamgage_network(
    runtime,
    write_file=write_reviewed_streamgage_network_file,
)
reviewed_streamgage_decision_table = reviewed_network_write.decision_table
reviewed_network_result = reviewed_network_write.result

display(reviewed_streamgage_decision_table)
display(pd.Series(reviewed_network_result, name="reviewed_streamgage_network"))


## USGS Reviewed Discharge Records

After the reviewed streamgage network exists, collect or reuse the discharge records for accepted active gages. These are the inland analogue to Marshfield's observed coastal boundary record.


In [ ]:
reviewed_records_result = collect.collect_or_reuse_reviewed_streamflow_records(
    runtime,
    skip_existing=source_skip_existing,
)

display(pd.Series(reviewed_records_result, name="usgs_reviewed_discharge_records"))


## Readiness Audit

The readiness report records which artifacts are present and which source contracts still need attention before event-catalog construction.


In [ ]:
readiness_summary, gates = collect.source_collection_readiness_report(config, paths)
display(readiness_summary)
display(gates)
display(collect.collection_readiness_table(runtime))


## Collected Data Overview

Plot the collected source data in the form that best matches each source: spatial footprints and sample points for geography, source-window bars for coverage, and time-series or distribution plots for the dynamic forcing records.


In [ ]:
# Set up small plotting helpers.
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def _location_path(value):
    path = Path(value)
    if path.is_absolute():
        return path
    if path.parts and path.parts[0] in {"data", "02_flood", "01_grid"}:
        return paths["location_root"] / path
    return paths["repo_root"] / path


def _read_csv(path, **kwargs):
    path = Path(path)
    return pd.read_csv(path, **kwargs) if path.exists() else pd.DataFrame()


def _first_column(frame, names):
    return next((name for name in names if name in frame.columns), None)


display(collect.collected_data_overview())


### SST

Storm transposition targets are plotted against the configured SST region and study area.


In [ ]:
# Plot the SST region, study area, and rainfall transposition targets.
import geopandas as gpd

study_area = gpd.read_file(_location_path(config["grid_footprint"]["source"])).to_crs("EPSG:4326")
sst_region = gpd.read_file(
    _location_path(config["collection"]["aorc_sst"]["transposition_region"]["geometry_file"])
).to_crs("EPSG:4326")
rainfall = _read_csv(paths["aorc_sst_rainfall_members_csv"], parse_dates=["storm_start", "storm_end"])

lon_column = _first_column(rainfall, ["centroid_lon", "transposed_centroid_lon"])
lat_column = _first_column(rainfall, ["centroid_lat", "transposed_centroid_lat"])
value_column = _first_column(rainfall, ["max_precip_in", "mean_precip_in", "max", "mean"])

display(pd.Series({
    "study_area_features": len(study_area),
    "sst_region_features": len(sst_region),
    "rainfall_rows": len(rainfall),
    "longitude_column": lon_column,
    "latitude_column": lat_column,
    "value_column": value_column,
}, name="sst_plot_inputs"))

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
sst_region.plot(ax=ax, facecolor="#f4a26133", edgecolor="#d95f02", linewidth=1.8)
study_area.boundary.plot(ax=ax, color="black", linewidth=1.2)

if not rainfall.empty and lon_column and lat_column:
    rainfall_points = gpd.GeoDataFrame(
        rainfall.dropna(subset=[lon_column, lat_column]),
        geometry=gpd.points_from_xy(rainfall[lon_column], rainfall[lat_column]),
        crs="EPSG:4326",
    )
    rainfall_points.plot(
        ax=ax,
        column=value_column,
        cmap="inferno_r",
        markersize=24,
        alpha=0.75,
        edgecolor="white",
        linewidth=0.15,
        legend=value_column is not None,
        legend_kwds={"label": "72h rainfall magnitude", "shrink": 0.62},
    )

ax.legend(
    handles=[
        Patch(facecolor="#f4a26133", edgecolor="#d95f02", label="AORC SST region"),
        Patch(facecolor="none", edgecolor="black", label="study area"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#c51b7d", markersize=8, label="rainfall transposition targets"),
    ],
    loc="best",
)
ax.set_title("Collected source geography")
ax.set_xlabel("")
ax.set_ylabel("")
plt.show()


### USGS Streamgages

Reviewed streamgages are plotted against the same source geography used during the review gate: evaluation footprint, SFINCS coverage, Wflow watershed search domain, and SST region.


In [ ]:
# Plot reviewed and candidate USGS streamgages over the source review geography.
area_layer_specs = collect.streamgage_review_area_layer_specs(runtime)
display(pd.DataFrame(area_layer_specs)[["label", "path", "edgecolor", "linestyle"]])

if runtime.candidate_path.exists() and runtime.reviewed_network_path.exists():
    streamgage_qa = collect.plot_streamgage_review_regions(runtime, area_layer_specs)
    display(streamgage_qa.artifact_summary)
    display(streamgage_qa.gage_domain_summary)
else:
    display(pd.Series({
        "status": "missing streamgage plot inputs",
        "candidate_gages": str(runtime.candidate_path),
        "candidate_gages_exist": runtime.candidate_path.exists(),
        "reviewed_network": str(runtime.reviewed_network_path),
        "reviewed_network_exists": runtime.reviewed_network_path.exists(),
    }, name="usgs_streamgage_plot"))


### NWM Soil Moisture

Soil moisture is summarized across configured NWM points and layers.


In [ ]:
# Plot monthly mean NWM soil moisture variables when available.
soil_moisture = _read_csv(paths["nwm_soil_moisture_csv"], parse_dates=["time"])
requested_soil_variables = list(config["collection"].get("nwm", {}).get("soil_moisture", {}).get("variables", []))
available_soil_variables = [name for name in requested_soil_variables if name in soil_moisture.columns]
legacy_soil_variable = _first_column(soil_moisture, ["SOIL_M", "soil_m", "soil_moisture"])
if legacy_soil_variable and legacy_soil_variable not in available_soil_variables:
    available_soil_variables.append(legacy_soil_variable)
missing_soil_variables = [name for name in requested_soil_variables if name not in soil_moisture.columns]

if not soil_moisture.empty and available_soil_variables:
    monthly = (
        soil_moisture.groupby("time")[available_soil_variables]
        .mean()
        .resample("MS")
        .mean()
    )
    fig, ax = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
    monthly.plot(ax=ax, linewidth=0.9)
    ax.set_title("NWM soil moisture state")
    ax.set_xlabel("")
    ax.set_ylabel("soil moisture / saturation")
    ax.legend(title="variable", loc="best", fontsize=8)
    plt.show()

display(pd.Series({
    "requested_variables": requested_soil_variables,
    "available_variables": available_soil_variables,
    "missing_variables": missing_soil_variables,
    "csv": str(paths["nwm_soil_moisture_csv"]),
}, name="nwm_soil_moisture_status"))


### AORC SST Rainfall

Rainfall members are shown as selected-event magnitude and distribution summaries.


In [ ]:
# Plot compact AORC SST rainfall member summaries.
rainfall = _read_csv(paths["aorc_sst_rainfall_members_csv"], parse_dates=["storm_start", "storm_end"])

display(pd.Series({
    "rainfall_rows": len(rainfall),
    "rainfall_members": str(paths["aorc_sst_rainfall_members_csv"]),
}, name="aorc_sst_rainfall_plot_inputs"))

if not rainfall.empty:
    max_column = _first_column(rainfall, ["max_precip_mm", "max_precip_in", "max"])
    mean_column = _first_column(rainfall, ["mean_precip_mm", "mean_precip_in", "mean"])
    if max_column is None or mean_column is None:
        raise KeyError("rainfall members need max and mean precipitation columns")
    unit = "mm"
    if "precip_units" in rainfall and rainfall["precip_units"].notna().any():
        unit = str(rainfall["precip_units"].dropna().iloc[0])
    elif str(max_column).endswith("_in"):
        unit = "in"
    fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)
    rainfall.plot.scatter(x="rank", y=max_column, ax=axes[0], color="#d95f02", s=16, alpha=0.7)
    axes[0].set_title("AORC SST rainfall member maxima")
    axes[0].set_xlabel("rank")
    axes[0].set_ylabel(f"precipitation, {unit}")
    rainfall[[mean_column, max_column]].plot.hist(ax=axes[1], bins=20, alpha=0.65)
    axes[1].set_title("Rainfall distribution")
    axes[1].set_xlabel(f"precipitation, {unit}")
    plt.show()
